In [2]:
import numpy as np
import math
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time
import itertools

In [3]:
n = 6
m_star = 3 

#creating actions
def create_actions(n):
    n_site = 2*n-3
    d_array = np.linspace(1,(2)**(n_site),(2)**(n_site))
    d_array = d_array.astype(int)
    # decimal to base 2, since action can be 0 or 1
    power=np.ones((d_array.shape[0],1))*((2)**np.arange(n_site)[::-1])
    m_array = np.floor((d_array[:,None]%((2)*power))/power) #0 to n-1 indices request for links (whether or not) and n to 2*n-3 (n-2 indices in total have swap requests)
    
    #removing all actions that have request and swap request at same node
    u=0
    for m in m_array:
        flag=0
        for k in range(n-1,len(m)):
            if m[k]==1:
                if m[k-(n-1)]==1 or m[k-(n-1)+1]==1:
                    flag=1
                    break
        if flag==1:
            m_array = np.delete(m_array,u,axis=0)
            u-=1
        u+=1

    actions = [None]*len(m_array)
    #converting to matrix form
    for x in range(len(m_array)):
        action = np.zeros([n,n])
        for i in range(n-1):
            action[i][i+1] = m_array[x][i]
            action[i+1][i] = m_array[x][i]
            if i>0 and i<n-1:
                action[i][i] = m_array[x][i+n-2]
        actions[x] = action
    return(actions)

all_actions = create_actions(n)
all_actions = np.unique(all_actions, axis=0)


from sympy.utilities.iterables import multiset_permutations

numbers = list(range(1, n))
result = []

def allowed_perms(item):
    allowed = True
    for i in range(n):
        if item[i]<=i and item[i]!=0:
            allowed = False
            break
    if allowed:
        for i in range(n):
            if item[i]>i+1:
                for j in range(i+1, item[i]):
                    if item[j] > item[i]:
                        allowed = False
                        break
    return allowed

for i in range(n):
    subsets = itertools.combinations(numbers, i)
    for subset in subsets:
        zero_count = n - len(subset)
        temp = list(subset) + [0]*zero_count
        temp_list = [None]
        
        temp_list = multiset_permutations(temp)
        
        if temp_list:
            for zero_added_perms in temp_list:
                if zero_added_perms:
                    if allowed_perms(zero_added_perms):
                        result.append(list(zero_added_perms))


def all_states_nodes(nodes):
    nodes = np.array(nodes)
    n_act_l = nodes[nodes>0]
    n_act_r = np.where(nodes>0)[0]
    n_a = len(n_act_l)

    d_array = np.linspace(1,(m_star+1)**n_a-1,(m_star+1)**n_a-1)
    d_array = d_array.astype(int)
    
    #converting state number in decimal to m_star base number produces the state 
    power=np.ones((d_array.shape[0],1))*((m_star+1)**np.arange(n_a)[::-1])
    m_array = np.floor((d_array[:,None]%((m_star+1)*power))/power) #m_array[i][j] has state of jth link of the ith possible state
    
    states = [None]*len(m_array)
    
    for x in range(len(m_array)):
    #writing the states in the matrix form
#         print(m_array[x])
        state = np.zeros([n,n])
        if n_a>0:
            for i in range(n_a):
                state[n_act_r[i]][n_act_l[i]] = m_array[x][i]
                state[n_act_l[i]][n_act_r[i]] = m_array[x][i]
#             print(state)
            states[x] = state
            
#     print("states")
#     print(states)
    return(np.unique(states,axis=0))

all_st = []
for k in result:
    if len(all_states_nodes(k))>0:
#         print(len(all_states_nodes(k)))
        all_st.append(all_states_nodes(k)) 
        
all_states=[]
for i in all_st:
    for j in i:
        all_states.append(j)

all_states.append(np.zeros([n,n]))
all_states = np.unique(all_states,axis=0) 
print(len(all_states))

3304


In [4]:
print([len(all_states), len(all_actions)])

[3304, 89]


In [27]:
D1_s = []
D2_s = []
for state in all_states:
    D1_s.append(np.linalg.eig(state)[0][0])
    D2_s.append(np.linalg.eig(state)[0][1])  
    
D1_s=np.array(D1_s)
D2_s=np.array(D2_s)

D1_a = []
D2_a = []
for action in all_actions:
    D1_a.append(np.linalg.eig(action)[0][0])
    D2_a.append(np.linalg.eig(action)[0][1])  
    
D1_a=np.array(D1_a)
D2_a=np.array(D2_a)

In [28]:
n_state = len(all_states)
n_action = len(all_actions)
q_values = np.zeros([n_state,n_action])

In [63]:
# Train the Model
# Our next task is for our AI agent to learn about its environment by implementing a Q-learning model. 
# The learning process will follow these steps:
# Choose a random, non-terminal state for the agent to begin this new episode.
# Choose an action for the current state. Actions will be chosen using an epsilon greedy algorithm. 
# This algorithm will usually choose the most promising action for the AI agent, 
# but it will occasionally choose a less promising option in order to encourage the agent to explore the environment.
# Perform the chosen action, and transition to the next state (i.e., move to the next location).
# Receive the reward for moving to the new state, and calculate the temporal difference.
# Update the Q-value for the previous state and action pair.
# If the new (current) state is a terminal state, go to #1. Else, go to #2.
# This entire process will be repeated across 1000 episodes. 
# This will provide the AI agent sufficient opportunity to learn the shortest paths 
# Define Helper Functions
def is_allowed_state(state):
    i=0
    res = False
    subset_states = np.nonzero(np.isclose(np.linalg.eig(state)[0][0],D1_s))[0]
    while not res and i<len(subset_states):
        res = np.array_equal(state, all_states[subset_states[i]])
        i+=1 
    return res
#     res = [np.array_equal(state, i) for i in all_states]
#     if  np.argwhere(np.array(res)==True).size==0:
#         return False
#     else:
#         return True

#define a function that determines if the specified location is a terminal state
def is_terminal_state(current_state):
  #if the state has a link between 1st and last node then it is terminal
    if current_state[0][n-1] > 0 and is_allowed_state(current_state):  
        return True
    else:
        return False

def get_state_index(state):
    i=0
    res = False
    subset_states = np.nonzero(np.isclose(np.linalg.eig(state)[0][0],D1_s))[0]
    while not res:
        res = np.array_equal(state, all_states[subset_states[i]])
        i+=1 
    return subset_states[i-1]

# def get_state_index(state):
# #     res = [np.array_equal(state, i) for i in all_states]
# #     index = np.argwhere(np.array(res)==True)[0][0]
#     i=0
#     res = False
#     while not res:
#         res = np.array_equal(state, all_states[i])
#         i+=1 
#     return i-1

def get_action_index(action):
    i=0
    res = False
    subset_actions = np.nonzero(np.isclose(np.linalg.eig(action)[0][0],D1_a))[0]
    while not res:
        res = np.array_equal(action, all_actions[subset_actions[i]])
        i+=1 
    return subset_actions[i-1]

# def get_action_index(action):
#     i=0
#     res = False
#     while not res:
#         res = np.array_equal(action, all_actions[i])
#         i+=1 
#     return i-1

def get_random_state():
    i = np.random.randint(0,len(all_states))
    return all_states[i]

def get_random_action():
    i = np.random.randint(0,len(all_actions))
    return all_actions[i]

#define a function that will choose a random, non-terminal starting state
def get_starting_state():
    
#   get a random state
    current_state = get_random_state()

#   continue choosing random state until a non-terminal state is identified
    while is_terminal_state(current_state):
        current_state = get_random_state()

    return current_state


#define an epsilon greedy algorithm that will choose which action to take next
def get_next_action(current_state, epsilon):
  #if a randomly chosen value between 0 and 1 is less than epsilon, 
  #then choose the most promising value from the Q-table for this state.
    state_index = get_state_index(current_state)
    
    if np.random.random() < epsilon: #return opt_action 
        action_index = np.argmax(q_values[state_index])
        opt_action = all_actions[action_index]
        return opt_action
    else: #choose a random action
        return get_random_action()

#define a function that will get the next location based on the chosen action
def get_next_state(current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n,n])
    
    for i in range(n):
        for j in range(n):
            new_state[i][j] = current_state[i][j]
    
    flag=0
    for i in range(n-1):
        if action[i][i+1]>0:
            flag=1
            break
            
    if flag==1:
        # wait                

        for i in range(n-1):
            for j in range(i+1,n):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                new_state[j][i] = new_state[i][j] 
            
    for i in range(n-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            if np.random.random()<=p_l:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0        
        
        for k in range(i+2,n):
            if new_state[i][k]>0:
                new_state[i][k] = 0
        
        for k in range(0,i):
            if new_state[i+1][k]>0:
                new_state[i+1][k] = 0

        new_state[j][i] = new_state[i][j]   
    
    
    #Bell measurements
    bell_flag = 0
    for i in range(1,n-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = max(node1_val,node2_val)
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
            else:
                if flag1==1 and flag2==0:
                    new_state[i][node1] = 0
                if flag2==1 and flag1==0:
                    new_state[node2][i] = 0
                    
        for i in range(n):
            for j in range(i+1,n):
                new_state[j][i] = new_state[i][j]
            
    return new_state

In [64]:
# Reward function
def rewards(state):
    if is_terminal_state(state):
        reward = 100
    else:
        reward = -1            
    return reward

In [65]:
def training_network(epsilon, discount_factor, learning_rate, p_l, p_bm, total_episodes):
    
    #run through 'total_episodes' training episodes
    for episode in range(int(total_episodes)):
        if episode%1000==0:
            print(episode)
#             if epsilon<0.8:
#                 epsilon+=0.05
        
        #get the starting state for this episode
        state = get_starting_state()
        
        
        #continue taking actions until we reach a terminal state or we do 200 actions
        trial_num = 0
#         t0= time.time()
        while trial_num<200 and is_terminal_state(state)==False:
            
            
            #choose which action to take
            
            action = get_next_action(state, epsilon)
            
            #perform the chosen action, and transition to the next state
            old_state = np.zeros([n,n]) #store the old state
            for i in range(n):
                for j in range(n):
                    old_state[i][j] = state[i][j]
             
            state = get_next_state(old_state, action, p_l, p_bm)
            
            
            if not is_allowed_state(state):
                state = np.zeros([n,n])
                 
            old_state_index = get_state_index(old_state)
            state_index = get_state_index(state)
            action_index = get_action_index(action)
            
            
            #receive the reward for moving to the new state, and calculate the temporal difference
            
            reward = rewards(state)
            
            
            old_q_value = q_values[old_state_index][action_index]
            temporal_difference = reward + (discount_factor * np.max(q_values[state_index])) - old_q_value

            #update the Q-value for the previous state and action pair
            new_q_value = old_q_value + (learning_rate * temporal_difference)
            q_values[old_state_index][action_index] = new_q_value
            trial_num = trial_num + 1
        
        if episode%4000==0:
            np.save("q_valuesn6m3_pl0.5_pbm0.75_wt_episodic_"+str(episode)+".npy",q_values)
#         print(time.time()-t0)     
    print('Training complete!')

In [67]:
#parallel learning
from copy import copy, deepcopy
def trajectories(state,q_values):
#     t0 = time.time()
    q_values_temp = np.zeros([n_state,n_action])
    q_values_temp = deepcopy(q_values)
#     for i in range(n_state):
#         for j in range(n_action):
#             q_values_temp[i][j] = q_values[i][j]
    
    #continue taking actions until we reach a terminal state or we do 200 actions
    trial_num = 0

    while trial_num<200 and is_terminal_state(state)==False:
        #print(trial_num)
        #choose which action to take

        action = get_next_action(state, epsilon)

        #perform the chosen action, and transition to the next state
        old_state = np.zeros([n,n]) #store the old state
        for i in range(n):
            for j in range(n):
                old_state[i][j] = state[i][j]

        state = get_next_state(old_state, action, p_l, p_bm)

        if not is_allowed_state(state):
            state = np.zeros([n,n])

        old_state_index = get_state_index(old_state)
        state_index = get_state_index(state)
        action_index = get_action_index(action)

        #receive the reward for moving to the new state, and calculate the temporal difference

        reward = rewards(state)

        old_q_value = q_values_temp[old_state_index][action_index]
        temporal_difference = reward + (discount_factor * np.max(q_values_temp[state_index])) - old_q_value

        #update the Q-value for the previous state and action pair
        new_q_value = old_q_value + (learning_rate * temporal_difference)
        q_values_temp[old_state_index][action_index] = new_q_value
        trial_num = trial_num + 1
        
#     print(time.time()-t0)        
    return q_values_temp
    
def training_network_parallel(epsilon, discount_factor, learning_rate, q_values):
    
    #run through 500 training episodes
    for episode in range(int(total_episodes)):
        if episode%10==0:
            print(episode)
#             if episode>1000 and epsilon<0.9:
#                 epsilon+=0.05
        
        #get the starting state for this episode
        state = get_starting_state()
#         print(state)
        q_values_list = Parallel(n_jobs=40)(delayed(trajectories)(state,q_values) for i in range(40))
        q_values = np.mean(q_values_list, axis=0)
        if episode%1000==0:
            np.save("q_valuesn6m3_pl0.5_pbm0.75_wt_parallel_" + str(episode) + ".npy", q_values)    
    print('Training complete!')

In [ ]:
total_episodes = 10000
learning_rate = 0.01
p_l = 0.5  
p_bm = 0.75
epsilon = 0.15
discount_factor = 0.8

q_values = np.zeros([n_state,n_action])

t0=time.time()
training_network_parallel(epsilon, discount_factor, learning_rate, q_values)
print(time.time()-t0)

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
1450
1460
1470
1480
1490
1500
1510
1520
1530
1540
1550
1560
1570
1580
1590
1600
1610
1620
1630
1640
1650
1660
1670
1680
1690
1700
1710
1720
1730
1740
1750
1760
1770
1780
1790
1800
1810
1820
1830
1840
1850
1860
1870
1880
1890
1900
1910
1920
1930
1940
1950
1960
1970
1980
1990
2000
2010
2020
2030
2040
2050
2060
2070
2080
2090
2100
2110
2120
2130
2140
2150
2160
2170
2180
2190
2200
2210
2

In [68]:
#Define a function that will get the shortest path between any initial state and final state.
def evolve_state(start_state, p_l, p_bm, steps, printing=False):
    trial_num = 0
    #if this is a 'legal' starting state
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
    while trial_num<steps:
        #get the best action to take
        action = get_next_action(current_state, 1.)
        if printing:
            print(np.array(action))
            print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
        if printing:
            print(np.array(current_state))
            print("")
        evolution_path.append(current_state)
        trial_num = trial_num + 1
    return evolution_path

def evolve_sa(start_state, p_l, p_bm, steps, printing=False):
    trial_num = 0
    #if this is a 'legal' starting state
    current_state = np.array([i for i in start_state])
    evolution_path = []
    action_path = []
#     evolution_path.append(current_state)
    #continue moving along the path until we reach the goal (i.e., all active node state) or 5000 actions
    while trial_num<steps:
        evolution_path.append(current_state)
        #get the best action to take
        action = get_next_action(current_state, 1.)
        action_path.append(action)
        if printing:
            print(np.array(action))
            print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
        if printing:
            print(np.array(current_state))
            print("")
#         evolution_path.append(current_state)
        trial_num = trial_num + 1
    return evolution_path,action_path

In [83]:
def shortest_path(start_state, p_l, p_bm, cc=False, nreq=False):
    
    current_state = np.array([i for i in start_state])
    evolution_path = []
    evolution_path.append(current_state)
    nr=0
    trial_num=0
    #continue moving along the path until we reach the goal (i.e., all active node state)
    while not is_terminal_state(current_state):
        #get the best action to take
        action = get_next_action(current_state, 1.)
        
        flag=0
        for i in range(n-1):
            if action[i][i+1]>0:
                flag=1
                break
        
        if nreq==True:
            for i in range(n-1):
                if action[i][i+1]>0:
                    nr+=1
#         print(action)
#         print("")
        
        #move to the next location on the path, and add the new location to the list
        current_state = np.array([i for i in get_next_state(current_state, action, p_l, p_bm)])
#         print(current_state)
#         print("")
        if cc==False:
            if flag==1:
                evolution_path.append(current_state)
        else:
            evolution_path.append(current_state)
        trial_num+=1
    evolution_path.append(current_state)    
    if nreq==False:    
        return evolution_path
    else:
        return evolution_path,nr

In [90]:
n = 6
m_star = 3
p_l = 0.5
p_bm = 0.75
q_values = np.load("q_valuesn6m3_pl0.5_pbm0.75_wt_episodic_"+str(20000)+".npy", allow_pickle=True)
shortest_path(np.zeros([n, n]), p_l, p_bm, cc=False, nreq=False)[-1]

array([[0., 0., 0., 0., 0., 2.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [2., 0., 0., 0., 0., 0.]])

In [91]:
#average waiting time
n = 6
m_star = 3
mean_list=[]
mean_list_fid=[]

p_l = 0.5
p_bm = 0.75
for episode in [30000]:
    q_values = np.load("q_valuesn6m3_pl0.5_pbm0.75_wt_episodic_"+str(episode)+".npy",allow_pickle=True)

    mean_mean=[]
    mean_fid=[]
    for p in range(10):
        print(p)
        K_n = []
        K_n_fid = []
        for k in range(100):
            print(k)
            state = np.zeros([n, n])
            K_n.append(len(shortest_path(state, p_l, p_bm, cc=False, nreq=False))-2)
            K_n_fid.append(shortest_path(state, p_l, p_bm, cc=False, nreq=False)[-1][0][n-1])

        mean_mean.append(np.mean(K_n))
        mean_fid.append(np.mean(K_n_fid))

    print(np.mean(mean_mean),np.std(mean_mean))
    print(np.mean(mean_fid))
    mean_list.append(np.mean(mean_mean))
    mean_list_fid.append(np.mean(mean_fid))

0
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
2
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
3
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
